# MUSSA: train the coronary segment model on ARCADE

Runs on Google Colab or Kaggle with a free GPU. It trains only on ARCADE, a public CC0 dataset (https://zenodo.org/records/10390295). No patient data is used or uploaded.

**Colab:** Runtime > Change runtime type > GPU.
**Kaggle:** in the notebook settings turn on the GPU accelerator and Internet, and add `mussa-training-kit.zip` as a dataset (Add data > Upload).

Steps: install, download and verify ARCADE, load the kit, train, check the exported model, download two files and copy them to `C:\Users\<you>\MUSSA-data\models\`.

Educational research model. Not a certified medical device.

In [ ]:
!pip -q install segmentation-models-pytorch onnx onnxscript onnxruntime

In [ ]:
import hashlib, os, pathlib, urllib.request, zipfile

ON_COLAB = pathlib.Path("/content").exists()
BASE = pathlib.Path("/content") if ON_COLAB else pathlib.Path("/kaggle/working")
URL = "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
EXPECTED_MD5 = "c5b1973ade06f7dff210f878161e1a76"

ARCADE_ZIP = BASE / "arcade.zip"
if not ARCADE_ZIP.exists():                     # re-running the cell does not download again
    urllib.request.urlretrieve(URL, ARCADE_ZIP)
digest = hashlib.md5()
with open(ARCADE_ZIP, "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_MD5, f"ARCADE download is corrupt or changed: {digest.hexdigest()}"
DATA = BASE / "arcade_data"
if not DATA.exists():
    zipfile.ZipFile(ARCADE_ZIP).extractall(DATA)
ARC = next(p.parent for p in DATA.rglob("syntax")
           if p.is_dir() and "__MACOSX" not in p.parts and (p / "train").is_dir())
print("ARCADE verified and unpacked at", ARC)

In [ ]:
KIT_DIR = BASE / "kit"
if ON_COLAB:
    from google.colab import files
    uploaded = files.upload()                   # choose mussa-training-kit.zip
    KIT = next(iter(uploaded))
else:                                           # Kaggle: the kit was added as a dataset
    KIT = next(pathlib.Path("/kaggle/input").rglob("mussa-training-kit.zip"))
zipfile.ZipFile(KIT).extractall(KIT_DIR)
print("kit unpacked:", sorted(p.name for p in KIT_DIR.iterdir()))

In [ ]:
# about an hour on a free GPU; add  --smoke  first for a one-minute pipeline check.
# The best weights are saved as segments_best.pt before the ONNX export, so a failed export loses nothing.
!cd {KIT_DIR} && python train_segmentation.py --arcade "{ARC}" --out {BASE}/out

In [ ]:
import json
import numpy as np
import onnxruntime

OUT = BASE / "out"
card = json.load(open(OUT / "segments.card.json"))
session = onnxruntime.InferenceSession(str(OUT / "segments.onnx"), providers=["CPUExecutionProvider"])
size = card["input_size"]
shape = session.run(None, {session.get_inputs()[0].name: np.zeros((1, 1, size, size), np.float32)})[0].shape
assert shape == (1, 25, size, size), f"unexpected model output shape {shape}"
print("model checked:", shape, "| validation mean Dice:", card["val_mean_dice"], "| test mean Dice:", card["test_mean_dice"])
if ON_COLAB:
    from google.colab import files
    files.download(str(OUT / "segments.onnx"))
    files.download(str(OUT / "segments.card.json"))
else:
    print("On Kaggle, open the Output tab of this notebook and download segments.onnx and segments.card.json from", OUT)